# derived_8.4-eval-1.1 — Evaluation of MoE Routing Strategies under Shared Global Backbone Paradigm

This experiment evaluates **Mixture-of-Experts ($K=2$) routing strategies** and baseline models on the Washington-only `derived_8.4` split (7 stations, 2023–2025 test set). All models reuse the **54-feature shared global backbone** discovered in `derived_8.4-feature-selection-2.0` and search per-regime add-only deltas ($c0, c1 \in \{0, 5, 10\}$).

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

EXP_DIR = Path(".")
df_summary = pd.read_csv(EXP_DIR / "metrics_summary.csv")
df_per_regime = pd.read_csv(EXP_DIR / "per_regime_metrics_summary.csv")
df_grid = pd.read_csv(EXP_DIR / "delta_grid_summary.csv")

with open(EXP_DIR / "selected_features.json", "r") as f:
    selected_meta = json.load(f)

print(f"Loaded {len(df_summary)} models in leaderboard.")

Loaded 8 models in leaderboard.


## Overall Model Leaderboard

The table below ranks all evaluated models by unweighted test $R^2$ across the 2023–2025 period (6,620 test samples across 7 WA stations).

In [2]:
leaderboard_cols = [
    "model_name", "strategy_name", "pooled_r2", "pooled_rmse", "pooled_ubrmse", "pooled_bias", "pooled_mae", "pooled_pearson"
]
print("### Overall Leaderboard (2023-2025 Test Set)")
print(df_summary[leaderboard_cols].to_markdown(index=False))

### Overall Leaderboard (2023-2025 Test Set)
| model_name                                  | strategy_name         |   pooled_r2 |   pooled_rmse |   pooled_ubrmse |   pooled_bias |   pooled_mae |   pooled_pearson |
|:--------------------------------------------|:----------------------|------------:|--------------:|----------------:|--------------:|-------------:|-----------------:|
| Clustering_V0_Full_k2 (Winner c0=0, c1=10)  | Clustering_V0_Full_k2 |    0.81496  |     0.0438196 |       0.043337  |    0.00648567 |    0.0337195 |         0.905594 |
| Clustering_V0_Full_k2 (Backbone c0=0, c1=0) | Clustering_V0_Full_k2 |    0.814334 |     0.0438936 |       0.0435155 |    0.00574933 |    0.0337738 |         0.904903 |
| Clustering_Dynamic_k2 (Winner c0=0, c1=0)   | Clustering_Dynamic_k2 |    0.786606 |     0.0470573 |       0.0461129 |    0.00938019 |    0.0361913 |         0.892172 |
| Global Single Model (54 Backbone)           | Global_Single         |    0.77923  |     0.0478636 |    

## Add-Only Per-Regime Delta Grid Search Analysis

This section analyzes how per-regime feature deltas ($c0, c1 \in \{0, 5, 10\}$) affect test $R^2$ across different routing strategies.

In [3]:
print("### Strategy x Delta Grid Summary (Pooled R²)")
grid_pivot = df_grid.pivot_table(
    index="strategy_name",
    columns=["cluster_0_count", "cluster_1_count"],
    values="pooled_r2"
)
print(grid_pivot.to_markdown())

### Strategy x Delta Grid Summary (Pooled R²)
| strategy_name         |   (0, 0) |   (0, 5) |   (0, 10) |   (5, 0) |   (5, 5) |   (5, 10) |   (10, 0) |   (10, 5) |   (10, 10) |
|:----------------------|---------:|---------:|----------:|---------:|---------:|----------:|----------:|----------:|-----------:|
| Clustering_Dynamic_k2 | 0.786606 | 0.77928  |  0.771133 | 0.776704 | 0.769378 |  0.761231 |  0.763459 |  0.756133 |   0.747986 |
| Clustering_V0_Full_k2 | 0.814334 | 0.814302 |  0.81496  | 0.814146 | 0.814113 |  0.814771 |  0.789072 |  0.789039 |   0.789697 |
| Seasonal_Binary_k2    | 0.769795 | 0.756122 |  0.764271 | 0.765988 | 0.752315 |  0.760463 |  0.756494 |  0.74282  |   0.750969 |
| Trained_Gating_k2     | 0.735474 | 0.735064 |  0.732764 | 0.726236 | 0.725826 |  0.723526 |  0.719924 |  0.719514 |   0.717214 |
| Univariate_G_API_k2   | 0.769632 | 0.767081 |  0.756354 | 0.756171 | 0.75362  |  0.742893 |  0.763862 |  0.761311 |   0.750584 |


## Per-Regime Performance Breakdown

Performance breakdown by test regime partition ($N, R^2, \text{RMSE}, \text{Bias}$) for winning model configurations.

In [4]:
per_regime_cols = [
    "strategy_name", "cluster", "n_train", "n_test", "r2", "rmse", "ubrmse", "bias", "mae"
]
print("### Per-Regime Performance Breakdown")
print(df_per_regime[per_regime_cols].to_markdown(index=False))

### Per-Regime Performance Breakdown
| strategy_name         |   cluster |   n_train |   n_test |       r2 |      rmse |    ubrmse |         bias |       mae |
|:----------------------|----------:|----------:|---------:|---------:|----------:|----------:|-------------:|----------:|
| Global_Single         |         0 |     14608 |     6620 | 0.77923  | 0.0478636 | 0.0466868 |  0.0105484   | 0.0370592 |
| Clustering_V0_Full_k2 |         0 |     10624 |     4817 | 0.80246  | 0.0444639 | 0.0436213 |  0.00861491  | 0.0359221 |
| Clustering_V0_Full_k2 |         1 |      3984 |     1803 | 0.844023 | 0.0420501 | 0.0420426 |  0.000797068 | 0.0278349 |
| Clustering_Dynamic_k2 |         0 |      7974 |     3717 | 0.820626 | 0.046568  | 0.0455177 |  0.00983445  | 0.0351836 |
| Clustering_Dynamic_k2 |         1 |      6634 |     2903 | 0.415307 | 0.0476764 | 0.0468574 |  0.00879855  | 0.0374816 |
| Univariate_G_API_k2   |         0 |      7304 |     3513 | 0.807786 | 0.0490092 | 0.0483873 |  0.007

## Yearly Performance Breakdown

Evaluation across test years 2023, 2024, and 2025.

In [5]:
yearly_cols = ["model_name", "pooled_r2", "year_2023_r2", "year_2024_r2", "year_2025_r2"]
print("### Year-by-Year R² Breakdown")
print(df_summary[yearly_cols].to_markdown(index=False))

### Year-by-Year R² Breakdown
| model_name                                  |   pooled_r2 |   year_2023_r2 |   year_2024_r2 |   year_2025_r2 |
|:--------------------------------------------|------------:|---------------:|---------------:|---------------:|
| Clustering_V0_Full_k2 (Winner c0=0, c1=10)  |    0.81496  |       0.822971 |       0.783256 |       0.83029  |
| Clustering_V0_Full_k2 (Backbone c0=0, c1=0) |    0.814334 |       0.8213   |       0.785101 |       0.828436 |
| Clustering_Dynamic_k2 (Winner c0=0, c1=0)   |    0.786606 |       0.759412 |       0.778981 |       0.818225 |
| Global Single Model (54 Backbone)           |    0.77923  |       0.750748 |       0.770077 |       0.813582 |
| Seasonal_Binary_k2 (Winner c0=0, c1=0)      |    0.769795 |       0.733203 |       0.761812 |       0.812083 |
| Univariate_G_API_k2 (Winner c0=0, c1=0)     |    0.769632 |       0.730946 |       0.76847  |       0.807666 |
| Baseline V0 (50 Feats)                      |    0.760447 |     